# ST 554 Homework 9: Siona Benjamin
In this homework, we will use Spark MLlib to different models and determine the best performing one. For this homework a dataset on abalone. In this dataset, various physical attributes of abalone are gathered, along with the number of rings seen in the abalone which indicates their age. The process of determining the number of rings of the abalone is a tedious task, so it would be helpful to determine the age of the abalone with other physical attributes that are outlined in the dataset. 

Therefore, in this homework, we will us the abalone dataset and use easily determine physical features of the abalone to determine the number of rings it has, indicating the age of the organism.

Our first step will be to import `pyspark` along with all the necessary modules and then create our spark session.

In [1]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, GBTRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Interaction, StringIndexer, Binarizer, VectorAssembler

In [2]:
#create spark session
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 23:19:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Splitting the Data, Metrics, and Models
### Reading in and Splitting the Data
Before we train our model we need to import our data. To do this, we'll use pandas to read in the csv file. Looking at the data, we can see we have nine paramters in our dataset, with one categorical parameter and 8 numerical parameters. 

In [3]:
#read in abalone data as a pandas dataframe
abalone_data = pd.read_csv("abalone.csv")
abalone_data.head()

,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


Next, we'll convert our pandas dataframe to a spark dataframe using `createDataFrame()`.

In [4]:
#convert pandas dataframe to spark dataframe 
abalone = spark.createDataFrame(abalone_data)
abalone.show(5)

+---+------+--------+------+------------+--------------+--------------+------------+-----+
|sex|length|diameter|height|whole_weight|shucked_weight|viscera_weight|shell_weight|rings|
+---+------+--------+------+------------+--------------+--------------+------------+-----+
|  M| 0.455|   0.365| 0.095|       0.514|        0.2245|         0.101|        0.15|   15|
|  M|  0.35|   0.265|  0.09|      0.2255|        0.0995|        0.0485|        0.07|    7|
|  F|  0.53|    0.42| 0.135|       0.677|        0.2565|        0.1415|        0.21|    9|
|  M|  0.44|   0.365| 0.125|       0.516|        0.2155|         0.114|       0.155|   10|
|  I|  0.33|   0.255|  0.08|       0.205|        0.0895|        0.0395|       0.055|    7|
+---+------+--------+------+------------+--------------+--------------+------------+-----+
only showing top 5 rows


Now that we have our abalone spark dataframe, we can split the data into a testing set and a training set in a 70:30 ratio. After splitting our dataset we see that our training set has 2897 records and our testing set has 1280 records.

In [5]:
#create training and testing datasets
train_df, test_df = abalone.randomSplit([0.7, 0.3], seed=42)

In [6]:
print(f"The training dataset has {train_df.count()} records")
print(f"The testing dataset has {test_df.count()} records")

The training dataset has 2897 records
The testing dataset has 1280 records


### Metric for Judging Models
To compare the performance of the proceeding models, we will use the root mean squared error (RMSE). The RMSE measures the average difference between the model's predicted values and the actual values. 
### Model Selection
In this homework, models from three different classes will be chosen. These will be Linear Regression, Decision Tree Regression, and Gradient-boosted Tree Regression. 

#### Linear Regression
In linear regression, a dependent variable (y) is modeled as a function of an independent variable (x). This is done by finding a line that best fits the data. The equation for linear regression is y = B0 + B1x1 + B2x2 + ... + e where y is the dependent variable and B0 is the y-intercept. B1, B2 and so on are the coefficients or weights of the regression model. Generally, fitting a linear regression model is centered around finding the set of coefficients that best model y as a function of the x variables. 

#### Decision Tree Regression
A decision tree regressor predicts numerical values using a tree-like structure. This model splits the data based on features that best reduce prediction error. The model starts at the root node, and then chooses a feature to split the data creating a child node. The data is divided until a stop condition is reached. Finally, a final predicted value is assigned to each leaf node.

#### Gradient-boosted Tree Regression 
Gradient boosting is a technique that builds a series of decision trees. Each new tree is aimed at correcting the errors of the previous trees. In a regression task, the final prediction is made by adding up the outputs from all the trees. 

## Model Fitting
Now that we have our dataset imported and split into a training and testing set, we can begin setting up pipelines for our models.

### Linear Regression
We'll start by fitting our linear regression model. The first step is to set up our transformations that will go into our pipeline. First, we'll convert the sex paramter using `StringIndexer`. This converts the string values in the sex column to numbers where 0 indicates male, 1 indicates female, and 2 indicates infant.

In [7]:
#set up indexer transformation
indexer = StringIndexer(inputCols = ["sex"], outputCols = ["sex_numeric"])
#indexerTrans will now have a .transform() method
indexerTrans = indexer.fit(abalone) 

For our next transformation, we'll set up an interaction parameter between the length of the abalone and the whole weight of it. 

In [8]:
#set up interaction transformation
interactionTrans = Interaction(inputCols=["length","whole_weight"], outputCol="vec1")

As our next transformation, we'll set up an SQL query to select a subset of our columns to use as features. We'll also use this transformation to set our variable of interest as label.

In [9]:
#set up sql transformation
sqlTrans = SQLTransformer(
    statement = """
                SELECT length, diameter, height, whole_weight, sex_numeric, vec1, rings as label FROM __THIS__
                """
)

Our last transformation will be the `VectorAssembler` to complile our predictive varialbes into a features column. 

In [10]:
#set up assembler 
assembler = VectorAssembler(inputCols = ["length", "diameter", "height", "whole_weight","sex_numeric", "vec1"], outputCol = "features", handleInvalid = 'keep')

Now that we have our transformations set up, we can move onto creating our grid for cross validation. The parameters we want to explore for linear regression are the regularization parameter `regParam` and the elastic net parameter `elasticNetParam`. `regParam` controls the trade-off between fitting the training data and keeping the model coefficients small. An ideal setting of this parameter will prevent overfitting. `elasticNetParam` control the mix between LASSO regularization and Ridge regularization which also controls the coefficients of the model.

In [11]:
#create linear regression model object
lr = LinearRegression()
#parameter grid for the linear regression model 
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.01, 0.04, 0.06, 0.1]) \
    .addGrid(lr.elasticNetParam, [0, 0.5, 0.8, 0.9, 1]) \
    .build()
#set up pipeline for linear regression
pipeline = Pipeline(stages = [indexerTrans, interactionTrans, sqlTrans, assembler, lr])

With our parameter grid set up, we can now create our cross validator passing in our pipeline, parameter grid, and evaluation method.

In [12]:
#set up cross validation test 
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [13]:
# Run cross-validation, and choose the best set of parameters.
cvModel = crossval.fit(train_df)

26/04/14 23:20:02 WARN Instrumentation: [dae3dc1c] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:20:06 WARN Instrumentation: [ffe02251] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:20:07 WARN Instrumentation: [99a45724] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:20:07 WARN Instrumentation: [ec6b5648] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:20:08 WARN Instrumentation: [30b7689f] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:20:22 WARN Instrumentation: [0dbb686a] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:20:22 WARN Instrumentation: [90e718ea] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:20:23 WARN Instrumentation: [6638c06c] regParam is zero, which might cause numerical instability and overf

To determine the performance of the linear regression model, we can use the test set to find the RMSE.

In [14]:
#determine rmse of the linear regression model on the test set
lr_rmse = RegressionEvaluator().evaluate(cvModel.transform(test_df))

### Decision Tree Regressor

Next, we'll use the same process to train a decision tree regressor model. For this model, we'll use the same set of transformations as the linear regression model. For our cross validation step, we'll prove the maximum depth or `maxDepth`. This determines the maximum number of splits allowed in the decision tree. 

In [17]:
#create decision tree regressor object
dtr = DecisionTreeRegressor()
#parameter grid for the decision tree regressor model 
paramGrid = ParamGridBuilder() \
    .addGrid(dtr.maxDepth, [3,5,7,9,11]) \
    .build()
#set up pipeline for model
pipeline = Pipeline(stages = [indexerTrans, interactionTrans, sqlTrans, assembler, dtr])

Just like before, we'll pass our new pipeline and parameter grid to `CrossValidator`. 

In [18]:
#create cross validation step
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [19]:
#run cross validation
cvModel = crossval.fit(train_df)

In [20]:
#determine rmse of decision tree model on test set
dtr_rmse = RegressionEvaluator().evaluate(cvModel.transform(test_df))

### Gradient-Boosted Tree Regressor

Now it's time to test our gradient-boosted tree regressor. For this model, we will keep the same transformations as above as well. We'll also explore the `maxDepth` paramter where it controls the amount of splits that can be made in each tree.

In [24]:
#create gradient boosted tree regressor model 
gbtr = GBTRegressor()
#parameter grid for the gradient boosted tree regressor
paramGrid = ParamGridBuilder() \
    .addGrid(gbtr.maxDepth, [3,5,7,9,11]).build()
#set up pipeline for model
pipeline = Pipeline(stages = [indexerTrans, interactionTrans, sqlTrans, assembler, gbtr])

In [25]:
#set up cross validation for model 
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [26]:
#run cross validation
cvModel = crossval.fit(train_df)

26/04/14 23:39:40 WARN DAGScheduler: Broadcasting large task binary with size 1008.9 KiB
26/04/14 23:39:41 WARN DAGScheduler: Broadcasting large task binary with size 1023.3 KiB
26/04/14 23:39:41 WARN DAGScheduler: Broadcasting large task binary with size 1040.7 KiB
26/04/14 23:39:42 WARN DAGScheduler: Broadcasting large task binary with size 1056.7 KiB
26/04/14 23:39:42 WARN DAGScheduler: Broadcasting large task binary with size 1039.1 KiB
26/04/14 23:39:42 WARN DAGScheduler: Broadcasting large task binary with size 1039.6 KiB
26/04/14 23:39:43 WARN DAGScheduler: Broadcasting large task binary with size 1040.6 KiB
26/04/14 23:39:43 WARN DAGScheduler: Broadcasting large task binary with size 1041.3 KiB
26/04/14 23:39:43 WARN DAGScheduler: Broadcasting large task binary with size 1043.4 KiB
26/04/14 23:39:44 WARN DAGScheduler: Broadcasting large task binary with size 1046.8 KiB
26/04/14 23:39:44 WARN DAGScheduler: Broadcasting large task binary with size 1053.1 KiB
26/04/14 23:39:45 WAR

KeyboardInterrupt: 

In [ ]:
#determine gbtr rmse with test set
gbtr_rmse = RegressionEvaluator().evaluate(cvModel.transform(test_df))

In [28]:
print("Model RMSE Values\n")
print(f"Linear Regression: {lr_rmse}\n")
print(f"Decision Tree Regression: {dtr_rmse}\n")
print(f"Gradient Boosted Tree Regression: N/A")

Model RMSE Values

Linear Regression: 2.455154524821782

Decision Tree Regression: 2.4779550652344935

Gradient Boosted Tree Regression: N/A


26/04/14 23:45:25 WARN DAGScheduler: Broadcasting large task binary with size 1143.8 KiB
26/04/14 23:45:26 WARN DAGScheduler: Broadcasting large task binary with size 1128.5 KiB
26/04/14 23:45:26 WARN DAGScheduler: Broadcasting large task binary with size 1129.1 KiB
26/04/14 23:45:26 WARN DAGScheduler: Broadcasting large task binary with size 1130.1 KiB
26/04/14 23:45:27 WARN DAGScheduler: Broadcasting large task binary with size 1130.9 KiB
26/04/14 23:45:27 WARN DAGScheduler: Broadcasting large task binary with size 1133.2 KiB
26/04/14 23:45:28 WARN DAGScheduler: Broadcasting large task binary with size 1136.6 KiB
26/04/14 23:45:28 WARN DAGScheduler: Broadcasting large task binary with size 1141.2 KiB
26/04/14 23:45:28 WARN DAGScheduler: Broadcasting large task binary with size 1148.3 KiB
26/04/14 23:45:29 WARN DAGScheduler: Broadcasting large task binary with size 1158.7 KiB
26/04/14 23:45:29 WARN DAGScheduler: Broadcasting large task binary with size 1174.3 KiB
26/04/14 23:45:30 WAR

After running our three models, we find that our Linear Regression model has a RMSE of 2.46, our Decision Tree model has an RMSE of 2.48, and our Gradient-Boosted Tree Regressor was too complex to run in a reasonable time. Given this information we can see that our Linear Regression model performed the best.